# Multi-source Sequence Fetch Example

This notebook demonstrates how to retrieve DNA, RNA, protein sequences, and PDB structure metadata from NCBI Entrez, UniProt, and RCSB PDB using **`run_sequence_fetch`**.

### Key Features:
- **ID-first resolution** -- Provide UniProt, GenBank, RefSeq, PDB, or Gene IDs for precise retrieval
- **Name+organism fallback** -- Resolves targets by gene name when IDs are not available
- **Multi-type requests** -- Fetch protein, genomic DNA, CDS, transcript RNA, pre-mRNA, and structures in one call
- **Batch mode** -- Process multiple targets with adaptive chunk splitting and retry logic

## Imports

In [1]:
from bio_programming_tools.tools.sequence_retrieval import (
    SequenceFetchConfig,
    SequenceFetchInput,
    run_sequence_fetch,
)

## 1. Fetch Protein + Structure by ID

Retrieve the lac repressor protein sequence from PDB and structure metadata by providing a `pdb_id` directly. ID-first resolution is the most reliable path.

### API Reference

**`SequenceFetchInput`**

| Field | Type | Description |
|---|---|---|
| `requests` | `List[SequenceFetchRequest]` | One or more retrieval requests; a single dict is accepted and auto-wrapped |

**`SequenceFetchRequest`** (key fields)

| Field | Type | Default | Description |
|---|---|---|---|
| `target_name` | `str` | *(required)* | Gene, protein, or RNA name to resolve |
| `organism` | `str` | *(required)* | Organism for disambiguation |
| `sequence_types` | `List[str]` | *(required)* | Any of: `protein`, `dna_genomic`, `dna_cds`, `rna_transcript`, `rna_premrna`, `structure` |
| `uniprot_id` | `Optional[str]` | `None` | UniProt accession override |
| `genbank_accession` | `Optional[str]` | `None` | GenBank accession override |
| `refseq_accession` | `Optional[str]` | `None` | RefSeq accession override |
| `pdb_id` | `Optional[str]` | `None` | PDB ID override |
| `gene_id` | `Optional[str]` | `None` | NCBI Gene ID override |
| `protein_id` | `Optional[str]` | `None` | NCBI protein accession override |
| `transcript_id` | `Optional[str]` | `None` | Transcript accession override |
| `genomic_coordinates` | `Optional[str]` | `None` | Coordinates as `accession:start-end:strand` |

**`SequenceFetchConfig`** (key parameters)

| Field | Type | Default | Description |
|---|---|---|---|
| `chunk_size` | `int` | `25` | Initial requests per chunk |
| `chunk_timeout_seconds` | `int` | `60` | Timeout per chunk before split/retry |
| `request_timeout_seconds` | `int` | `15` | Timeout per HTTP call |
| `strict_type_checks` | `bool` | `True` | Enforce ncRNA/protein mismatch checks |
| `ncbi_api_key` | `Optional[str]` | `None` | Optional NCBI API key for higher rate limits |

**`SequenceFetchOutput`**

| Field | Type | Description |
|---|---|---|
| `results` | `List[SequenceFetchResult]` | Per-request retrieval outcomes |
| `num_requests` | `int` | Total number of requests |
| `num_success` | `int` | Requests completed without warnings |
| `num_warning` | `int` | Requests completed with warnings |
| `num_completed` | `int` | `num_success + num_warning` |
| `num_failed` | `int` | Requests that failed |

In [2]:
# Fetch lacI protein + genomic DNA + structure using a PDB ID
inputs = SequenceFetchInput(
    requests=[
        {
            "request_id": "lacI_ecoli",
            "target_name": "lacI",
            "organism": "Escherichia coli",
            "sequence_types": ["protein", "dna_genomic", "structure"],
            "pdb_id": "1LBG",
        }
    ]
)

config = SequenceFetchConfig(chunk_size=10, chunk_timeout_seconds=45)
result = run_sequence_fetch(inputs, config)

item = result.results[0]
print(f"Status: {item.status}")
print(f"Sequences: {len(item.fetched_sequences)}, Structures: {len(item.fetched_structures)}")
print(f"Resolved IDs: {item.resolved_ids}")

# Inspect fetched sequences
for seq in item.fetched_sequences:
    preview = seq.sequence[:50] + ("..." if len(seq.sequence) > 50 else "")
    print(f"\n  {seq.sequence_type} [{seq.source_database}] {seq.accession}")
    print(f"    length={seq.length}, preview={preview}")

# Inspect fetched structures
for struct in item.fetched_structures:
    print(f"\n  PDB {struct.pdb_id}: {struct.method}, resolution={struct.resolution}")
    print(f"    title={struct.title}")

if item.warnings:
    print(f"\nWarnings:")
    for w in item.warnings:
        print(f"  - {w}")

Status: warning
Sequences: 2, Structures: 1
Resolved IDs: {'pdb_id': '1LBG', 'protein_id': 'Chains', 'gene_id': '945007', 'genbank_accession': 'NC_000913.3:c367510-366428'}

  protein [pdb] 1LBG
    length=360, preview=MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPN...

  dna_genomic [ncbi] NC_000913.3:c367510-366428
    length=1083, preview=GTGAAACCAGTAACGTTATACGATGTCGCAGAGTATGCCGGTGTCTCTTA...

  PDB 1LBG: X-RAY DIFFRACTION, resolution=4.8
    title=LACTOSE OPERON REPRESSOR BOUND TO 21-BASE PAIR SYMMETRIC OPERATOR DNA, ALPHA CARBONS ONLY

Warnings:
  - Nuccore genomic candidates lacked FASTA; used gene-locus genomic fallback
  - Multiple gene candidates found (5); using top hit 945007
  - Fetched gene-locus genomic interval NC_000913.3:366428-367510:-


## 2. Fetch Protein + Structure by Name

When no IDs are available, provide just a gene name and organism. The tool searches UniProt first, then falls back to NCBI. For structures, it resolves UniProt cross-references to PDB.

In [3]:
# Fetch TP53 protein + structure by name only (no IDs provided)
inputs = SequenceFetchInput(
    requests=[
        {
            "request_id": "tp53_human",
            "target_name": "TP53",
            "organism": "Homo sapiens",
            "sequence_types": ["protein", "structure"],
        }
    ]
)

result = run_sequence_fetch(inputs, SequenceFetchConfig())
item = result.results[0]

print(f"Status: {item.status}")
print(f"Resolved IDs: {item.resolved_ids}")

seq = item.fetched_sequences[0]
preview = seq.sequence[:50] + ("..." if len(seq.sequence) > 50 else "")
print(f"\nProtein [{seq.source_database}] {seq.accession}: length={seq.length}")
print(f"  preview={preview}")

struct = item.fetched_structures[0]
print(f"\nStructure: PDB {struct.pdb_id}, method={struct.method}")

if item.warnings:
    print(f"\nWarnings:")
    for w in item.warnings:
        print(f"  - {w}")

Status: warning
Resolved IDs: {'uniprot_id': 'P04637', 'pdb_id': '1A1U'}

Protein [uniprot] P04637: length=393
  preview=MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDI...

Structure: PDB 1A1U, method=SOLUTION NMR

Warnings:
  - Using resolved UniProt ID 'P04637' from protein fetch for structure lookup
  - Using first UniProt-linked PDB ID '1A1U' from cross references


## 3. Batch Requests

Process multiple targets in a single call. Each request is independent -- partial failures on one target do not block others. The tool uses adaptive chunk splitting to handle upstream timeouts gracefully.

In [4]:
# Batch: fetch proteins for three targets in one call
inputs = SequenceFetchInput(
    requests=[
        {
            "request_id": "lacI_ecoli",
            "target_name": "lacI",
            "organism": "Escherichia coli",
            "sequence_types": ["protein"],
            "uniprot_id": "P0A6X3",
        },
        {
            "request_id": "tp53_human",
            "target_name": "TP53",
            "organism": "Homo sapiens",
            "sequence_types": ["protein"],
            "uniprot_id": "P04637",
        },
        {
            "request_id": "insulin_human",
            "target_name": "INS",
            "organism": "Homo sapiens",
            "sequence_types": ["protein"],
            "uniprot_id": "P01308",
        },
    ]
)

result = run_sequence_fetch(inputs, SequenceFetchConfig())

print(f"Requests:  {result.num_requests}")
print(f"Success:   {result.num_success}")
print(f"Warning:   {result.num_warning}")
print(f"Failed:    {result.num_failed}")
print()

for item in result.results:
    seq = item.fetched_sequences[0] if item.fetched_sequences else None
    status_icon = {"success": "+", "warning": "~", "failed": "x"}[item.status]
    length = seq.length if seq else 0
    print(f"  [{status_icon}] {item.request_id}: {item.status}, length={length}")

Requests:  3
Success:   0
Failed:    0

  [~] lacI_ecoli: warning, length=102
  [~] tp53_human: warning, length=393
  [~] insulin_human: warning, length=110


## 4. Export Results

Save fetched sequences to files for downstream analysis.

### Supported formats:
- **FASTA** -- Standard FASTA with headers containing `request_id|target_name|sequence_type|source|accession`
- **JSON** -- Full structured output including metadata, warnings, and resolved IDs

In [5]:
from pathlib import Path

# Export to FASTA
result.export(name="batch_proteins", export_path="./example_output", file_format="fasta")
print("Exported to ./example_output/batch_proteins.fasta")

# Preview the FASTA content
fasta_path = Path("./example_output/batch_proteins.fasta")
print(f"\nFASTA preview:")
for line in fasta_path.read_text().splitlines()[:6]:
    print(f"  {line}")

# Export to JSON
result.export(name="batch_proteins", export_path="./example_output", file_format="json")
print("\nExported to ./example_output/batch_proteins.json")

Exported to ./example_output/batch_proteins.fasta

FASTA preview:
  >lacI_ecoli|lacI|protein|uniprot|P0A6X3
  MAKGQSLQDPFLNALRRERVPVSIYLVNGIKLQGQIESFDQFVILLKNTVSQMVYKHAISTVVPSRPVSHHSNNAGGGTSSNYHHGSSAQNTSAQQDSEETE
  >tp53_human|TP53|protein|uniprot|P04637
  MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD
  >insulin_human|INS|protein|uniprot|P01308
  MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN

Exported to ./example_output/batch_proteins.json
